In [1]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [ ]:
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 100        # ← set as desired
addprocs(num_workers)

4-element Vector{Int64}:
 2
 3
 4
 5

In [ ]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "data/median_RV.csv"
    data_column                  = "x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 2130
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "HAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "Epanechnikov"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "one-sided"
    kernel_type_tvEWD            = "Epanechnikov"
    kernel_type_tvHAR            = "Epanechnikov"
    kernel_type_tvAR             = "Epanechnikov"

    alpha_level                  = 0.05
end

In [6]:
const SED_PATH = abspath("src/SED_Thresholds/SEDThresholds.jl")

"c:\\Users\\mikul\\Desktop\\Persistence of Shocks\\tvPersistence.jl - Cleaned\\src\\SED_Thresholds\\SEDThresholds.jl"

In [7]:
@everywhere include($SED_PATH)        # <— absolute path shipped to workers
@everywhere using .SEDThresholds

In [8]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [9]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 3:	[ Info: Performing boostrap simulation number 2
      From worker 4:	[ Info: Performing boostrap simulation number 3
      From worker 5:	[ Info: Performing boostrap simulation number 4
      From worker 2:	[ Info: Performing boostrap simulation number 1
      From worker 4:	[ Info: Bootstrap 3 generated.
      From worker 2:	[ Info: Bootstrap 1 generated.
      From worker 3:	[ Info: Bootstrap 2 generated.
      From worker 5:	[ Info: Bootstrap 4 generated.


Task (done) @0x00000217d5a103f0

In [10]:
sed_vals

4-element Vector{Vector{Float64}}:
 [NaN, -0.0004271781315420682, -0.00014279089256275096, -3.542821439731007e-5, 3.634548139301873e-5, 5.824036074162094e-5, 6.371553061466071e-5, 4.827522509969791e-5, 4.804135043152123e-5, 4.6157747954340836e-5  …  7.448451010266582e-6, 7.73556907740348e-6, 7.633498419683455e-6, 7.931118613650344e-6, 7.869282925418393e-6, 7.45749682792073e-6, 7.11812964154115e-6, 7.0549303934175724e-6, 7.216607809726588e-6, 7.826858633942785e-6]
 [NaN, 4.181248283733227e-6, -1.8826484414948073e-6, 1.5221567895874537e-5, 1.1724470665287746e-5, 2.6296610989041777e-6, -2.929549484620543e-6, -6.006341522003077e-6, -8.477531051062012e-6, -8.8892032497351e-6  …  -6.471801834559104e-6, -5.989766166146206e-6, -5.662278208557362e-6, -5.5457633491576085e-6, -5.124884805014074e-6, -5.684154904975628e-6, -5.291897488117489e-6, -5.0053179002331936e-6, -4.73639680440722e-6, -4.390725131552138e-6]
 [NaN, 3.1239136154869277e-6, -7.320361431307812e-6, 7.0093476958949994e-6, -1.4431150

In [11]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

4

In [ ]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 2.755478427649564e-6


In [ ]:
# Save the SED values into BSON file
BSON.@save "sed_thresholds.bson" sed_vals thr